<a href="https://colab.research.google.com/github/VaibhavChandeliya/CAIC-COMPUTER-VISION/blob/main/CAIC_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
# Grayscale images have 1 channel, so we pass single values for mean and std
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
# Download and create the Training Dataset
train_set = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Download and create the Testing Dataset
test_set = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 171kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.11MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.8MB/s]


In [2]:
# Create DataLoaders for clean batching
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
# Grab a single batch from the loader
images, labels = next(iter(train_loader))

print("Batch Images Shape:", images.shape)
print("Batch Labels Shape:", labels.shape)


Batch Images Shape: torch.Size([64, 1, 28, 28])
Batch Labels Shape: torch.Size([64])


In [3]:
import torch
import torch.nn as nn

class FashionCustomMLP(nn.Module):
    def __init__(self):
        super().__init__()

        # Stacking the layers exactly to your description
        self.network = nn.Sequential(
            # 1. Flatten the 2D image (28x28) into a 1D vector (784 features)
            nn.Flatten(),

            # 2. Hidden Layer 1: 784 inputs -> 100 outputs
            nn.Linear(in_features=784, out_features=512),
            nn.ReLU(),

            # 3. Hidden Layer 2: 100 inputs -> 50 outputs
            nn.Linear(in_features=512, out_features=256),
            nn.ReLU(),

            # 4. Hidden Layer 3: 50 inputs -> 25 outputs
            nn.Linear(in_features=256, out_features=128),
            nn.ReLU(),

            # 5. Final Output Layer: 25 inputs -> 10 output neurons (one for each class)
            # NO ReLU here so we don't accidentally wipe out negative classification scores!
            nn.Linear(in_features=128, out_features=10)
        )

    def forward(self, x):
        # Pass the input batch directly through the sequential network chain
        return self.network(x)

# --- Instantiation and GPU Setup ---

# Detect if the Colab T4 GPU runtime is active
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Targeting Device: {device}")

# Instantiate your network and move its parameters onto the GPU VRAM
model = FashionCustomMLP().to(device)

# Print out the structural summary to verify your layer properties
print("\n--- Model Structural Architecture ---")
print(model)

Targeting Device: cuda

--- Model Structural Architecture ---
FashionCustomMLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Linear(in_features=256, out_features=128, bias=True)
    (6): ReLU()
    (7): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [4]:
import torch.nn as nn
import torch.optim as optim

# 1. Define the Loss Function (Handles the Softmax under the hood!)
loss_fn = nn.CrossEntropyLoss()

# 2. Define the Optimizer (Adam automatically updates weights using gradient descent)
# We pass model.parameters() so it knows which weights to tweak on your GPU
optimizer = optim.Adam(model.parameters(), lr=0.003)

print("Loss function and Optimizer successfully initialized!")


Loss function and Optimizer successfully initialized!


In [6]:
# Set your desired number of full passes over the dataset
epochs = 20

print(f"Starting Gradient Descent Loop for {epochs} Epochs on {device}...\n")

# The Outer Loop: Controls how many times the model sees the ENTIRE dataset
for epoch in range(epochs):

    # Track the accumulated loss score for printing status updates
    running_loss = 0.0

    # Set the model to explicit training mode
    model.train()

    # The Inner Loop: Fetches images in mini-batches of 64 from your train_loader
    for batch_idx, (images, labels) in enumerate(train_loader):

        # 1. Teleport your data batch onto the GPU VRAM
        images = images.to(device)
        labels = labels.to(device)

        # 2. CLEAR THE SLATE
        # Erase the calculus gradients from the previous batch of 64 images
        optimizer.zero_grad()

        # 3. THE FORWARD PASS
        # Feed the 64 images into the network to get 10 raw scores (logits) per image
        outputs = model(images)

        # 4. THE COST FUNCTION
        # Compute how wrong the model's guesses were (applies Softmax internally)
        loss = loss_fn(outputs, labels)

        # 5. THE BACKWARD PASS (Calculus)
        # Travel backward through the 4 layers to find the gradient/slope for every weight
        loss.backward()

        # 6. THE GRADIENT DESCENT STEP (The Weight Update)
        # Tell the Adam optimizer to tweak model.parameters() based on those slopes
        optimizer.step()

        # Accumulate the batch loss score
        running_loss += loss.item()

        # Print a progress report every 200 batches
        if batch_idx % 200 == 199:
            average_batch_loss = running_loss / 200
            print(f"Epoch [{epoch+1:02d}/{epochs}] | Batch [{batch_idx+1:03d}/{len(train_loader)}] | Current Loss: {average_batch_loss:.4f}")
            running_loss = 0.0

print("\n Optimization Engine Complete! Your network weights are fully trained.")

Starting Gradient Descent Loop for 20 Epochs on cuda...

Epoch [01/20] | Batch [200/938] | Current Loss: 0.2805
Epoch [01/20] | Batch [400/938] | Current Loss: 0.2804
Epoch [01/20] | Batch [600/938] | Current Loss: 0.2876
Epoch [01/20] | Batch [800/938] | Current Loss: 0.2896
Epoch [02/20] | Batch [200/938] | Current Loss: 0.2682
Epoch [02/20] | Batch [400/938] | Current Loss: 0.2805
Epoch [02/20] | Batch [600/938] | Current Loss: 0.2765
Epoch [02/20] | Batch [800/938] | Current Loss: 0.2891
Epoch [03/20] | Batch [200/938] | Current Loss: 0.2651
Epoch [03/20] | Batch [400/938] | Current Loss: 0.2666
Epoch [03/20] | Batch [600/938] | Current Loss: 0.2582
Epoch [03/20] | Batch [800/938] | Current Loss: 0.2845
Epoch [04/20] | Batch [200/938] | Current Loss: 0.2538
Epoch [04/20] | Batch [400/938] | Current Loss: 0.2628
Epoch [04/20] | Batch [600/938] | Current Loss: 0.2628
Epoch [04/20] | Batch [800/938] | Current Loss: 0.2545
Epoch [05/20] | Batch [200/938] | Current Loss: 0.2535
Epoch [0

In [7]:
# --- STEP 5: THE FINAL EXAM (TEST DATA EVALUATION) ---

# 1. Initialize scoring counters
correct_predictions = 0
total_images = 0

# 2. Set the model to evaluation mode
# (This tells PyTorch we are testing, not training)
model.eval()

# 3. Disable the calculus gradient engine to save memory and speed up the GPU
with torch.no_grad():

    # Loop through the 10,000 test images in batches of 64
    for images, labels in test_loader:

        # Teleport the test batch to your GPU
        images = images.to(device)
        labels = labels.to(device)

        # Forward Pass: Get the model's 10 raw scores per image
        outputs = model(images)

        # Use argmax to find the index of the highest score (the model's final guess)
        predictions = torch.argmax(outputs, dim=1)

        # Update our counters
        total_images += labels.size(0)
        correct_predictions += (predictions == labels).sum().item()

# 4. Compute and display the final percentage score
final_accuracy = (correct_predictions / total_images) * 100

print("=" * 50)
print(f" FINAL TEST ACCURACY: {final_accuracy:.2f}%")
print("=" * 50)

 FINAL TEST ACCURACY: 88.14%
